# Low-Chrome Mill Ball Solidification Simulation
**Module L99** | Physics-informed 1D radial FDM | Permanent metal mould

Run each cell in order by clicking the ▶ button. No installation required.

---

In [ ]:
# CELL 1 — Install dependencies and clone simulation code
!pip install numpy matplotlib pandas scipy scikit-learn tqdm ipywidgets --quiet
!git clone https://github.com/songeyargky/CastingSimulation.git
import sys
sys.path.insert(0, '/content/CastingSimulation')
print('Setup complete. Proceed to Cell 2.')

In [ ]:
# CELL 2 — Import simulation modules
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import config
import src.materials as mat
from src.grid import create_grid
from src.solver import update_temperature
from src.defects import (
    compute_misrun_risk, compute_cold_shut_risk,
    compute_surface_crack_index, compute_warpage_index
)
print('All modules loaded successfully.')

In [ ]:
# CELL 3 — Single run simulation with interactive controls
T_pour_w   = widgets.IntSlider(value=1450,min=1250,max=1550,step=10,description='T_pour (°C):',style={'description_width':'initial'},layout=widgets.Layout(width='500px'))
T_mold_w   = widgets.IntSlider(value=300,min=25,max=450,step=25,description='T_mold (°C):',style={'description_width':'initial'},layout=widgets.Layout(width='500px'))
diam_w     = widgets.Dropdown(options=[60,80,100,120],value=100,description='Diameter (mm):',style={'description_width':'initial'})
run_btn    = widgets.Button(description='▶  Run Simulation',button_style='success',layout=widgets.Layout(width='200px'))
out        = widgets.Output()

def run_sim(b):
    with out:
        clear_output(wait=True)
        print(f'Running: D={diam_w.value}mm  T_pour={T_pour_w.value}°C  T_mold={T_mold_w.value}°C ...')
        N = config.N_nodes
        r, dr, dt = create_grid(diam_w.value, N)
        T = np.ones(N) * T_pour_w.value
        surface_fs = 0.0
        time_val = 0.0
        next_save = 1.0
        time_pts, T_centre, T_surface, T_hist, fs_hist = [], [], [], [], []
        t_sol = None
        while time_val < config.max_sim_time:
            T, surface_fs = update_temperature(T,r,dr,dt,T_mold_w.value,h_initial=config.h_initial,h_gap=config.h_gap,time=time_val)
            time_val += dt
            if time_val >= next_save:
                time_pts.append(time_val); T_centre.append(T[0]); T_surface.append(T[-1])
                T_hist.append(T.copy()); fs_hist.append(mat.get_solid_fraction_profile(T))
                next_save += 1.0
                if t_sol is None and T[0] <= config.T_solidus:
                    t_sol = time_val
            if np.all(T - T_mold_w.value < 5.0): break
        MRI,_,_,_ = compute_misrun_risk(T_hist,fs_hist,time_pts,T_pour_w.value,config.h_initial)
        CSRI,_,_,_ = compute_cold_shut_risk(T_hist,time_pts,T_pour_w.value,T_mold_w.value)
        SCI,_,_,_ = compute_surface_crack_index(T_hist,fs_hist,time_pts)
        WI,_,_,_ = compute_warpage_index(T_hist,fs_hist,time_pts,r)
        WI_risk = float(np.clip(WI/0.010,0,1))
        fig, axes = plt.subplots(1,2,figsize=(14,5))
        axes[0].plot(time_pts,T_centre,label='T_centre',lw=2,color='#1f77b4')
        axes[0].plot(time_pts,T_surface,label='T_surface',lw=2,color='#ff7f0e')
        axes[0].axhline(config.T_liquidus,color='steelblue',ls=':',lw=1,label=f'T_liq={config.T_liquidus}°C')
        axes[0].axhline(config.T_solidus,color='steelblue',ls='--',lw=1,label=f'T_sol={config.T_solidus}°C')
        if t_sol: axes[0].axvline(t_sol,color='purple',ls=':',lw=1.5,label=f't_sol={t_sol:.0f}s')
        axes[0].set_xlabel('Time (s)'); axes[0].set_ylabel('Temperature (°C)')
        axes[0].set_title(f'Cooling Curve — {diam_w.value} mm Ball'); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.4)
        defects = ['Misrun','Cold Shut','Surf. Crack','Warpage']
        vals = [MRI,CSRI,SCI,WI_risk]
        colours = ['#2ecc71' if v<0.25 else '#f1c40f' if v<0.50 else '#e67e22' if v<0.75 else '#e74c3c' for v in vals]
        axes[1].barh(defects,vals,color=colours,edgecolor='#333',linewidth=0.8)
        for i,v in enumerate(vals): axes[1].text(v+0.01,i,f'{v:.3f}',va='center',fontsize=10)
        axes[1].set_xlim(0,1); axes[1].set_xlabel('Risk Index (0=safe, 1=very high)')
        axes[1].set_title('Defect Risk Summary'); axes[1].grid(alpha=0.3,axis='x')
        axes[1].axvline(0.25,color='green',ls='--',lw=0.8,alpha=0.6)
        axes[1].axvline(0.50,color='orange',ls='--',lw=0.8,alpha=0.6)
        axes[1].axvline(0.75,color='red',ls='--',lw=0.8,alpha=0.6)
        plt.tight_layout(); plt.savefig('result.png',dpi=120,bbox_inches='tight'); plt.show()
        print(f'\nSolidification time: {t_sol:.0f} s')
        print(f'Misrun (MRI):         {MRI:.3f}')
        print(f'Cold Shut (CSRI):     {CSRI:.3f}')
        print(f'Surface Crack (SCI):  {SCI:.3f}')
        print(f'Warpage (WI_risk):    {WI_risk:.3f}')
        print(f'Total Risk:           {MRI+CSRI+SCI+WI_risk:.3f}')

run_btn.on_click(run_sim)
display(T_pour_w, T_mold_w, diam_w, run_btn, out)

---
## Parameter Sweep (400 runs)
Run the next cell to execute the full factorial sweep and download results as CSV.

In [ ]:
# CELL 4 — Full parameter sweep (run time ~25 minutes on Colab)
import subprocess
import os
os.chdir('/content/millball-solidification-sim')
print('Starting sweep — this takes approximately 25 minutes...')
subprocess.run(['python', 'run_sweep.py'], check=True)
print('Sweep complete.')

# Download the CSV
from google.colab import files
files.download('sweep_results/parameter_sweep.csv')

In [ ]:
# CELL 5 — Run full analysis suite on sweep results
print('Generating all analysis figures...')
subprocess.run(['python', 'analyse_sweep.py'], check=True)
print('Analysis complete. Figures saved in sweep_results/')